# 🧩 Mini-Lab: Cost Tracking

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why tracking LLM API costs per request is critical for budget management
2. **Implement** a cost tracker that calculates dollar costs from token usage and model pricing
3. **Identify** which requests are most expensive by analyzing per-request cost breakdowns
4. **Recognize** how cost tracking integrates with observability to prevent budget surprises

## Target Concepts

| Concept | Description |
|---------|-------------|
| Cost Tracking | Monitoring the dollar cost of every LLM API call by combining token usage data with model pricing, enabling budget management and cost optimization |

## Setup

In [2]:
import time
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ Ready")

✅ Ready


## Why Track LLM Costs?

Every LLM API call costs money based on how many **tokens** you send (prompt) and receive (completion). Unlike traditional APIs with flat-rate pricing, LLM costs are **variable and unpredictable**:

- A short classification prompt might cost $0.0001
- A long document summarization might cost $0.01 — **100x more**
- A chatbot serving 10,000 users/day can accumulate hundreds of dollars before anyone notices

Without per-request cost tracking, you're flying blind. Cost tracking answers:

- **How much did each request cost?** → Identify expensive prompts
- **What's our daily/weekly spend?** → Budget forecasting
- **Which feature is most expensive?** → Optimize the right thing
- **Are costs trending up?** → Catch regressions early

In `mini-observability`, we logged token counts per request. Now we'll turn those token counts into **dollar amounts**.

## Step 1 — Define Model Pricing

LLM providers charge different rates for **input** (prompt) tokens and **output** (completion) tokens. We need a pricing table that maps model names to their per-token costs.

> **Note:** Prices change over time. Always check the provider's pricing page for current rates. The values below are representative examples.

In [3]:
# Pricing per token (in USD)
# Format: {model: {"input": price_per_token, "output": price_per_token}}
MODEL_PRICING = {
    "gpt-4o-mini": {
        "input":  0.15 / 1_000_000,   # $0.15 per 1M input tokens
        "output": 0.60 / 1_000_000,   # $0.60 per 1M output tokens
    },
    "gpt-4o": {
        "input":  2.50 / 1_000_000,   # $2.50 per 1M input tokens
        "output": 10.00 / 1_000_000,  # $10.00 per 1M output tokens
    },
}


def calculate_cost(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    """Calculate the dollar cost of an LLM call from token counts."""
    pricing = MODEL_PRICING.get(model)
    if pricing is None:
        return 0.0  # Unknown model — can't calculate
    input_cost = prompt_tokens * pricing["input"]
    output_cost = completion_tokens * pricing["output"]
    return input_cost + output_cost


# Quick check
example_cost = calculate_cost("gpt-4o-mini", prompt_tokens=100, completion_tokens=50)
print(f"Example: 100 input + 50 output tokens on gpt-4o-mini = ${example_cost:.6f}")
print("✅ Pricing table and calculate_cost() defined")

Example: 100 input + 50 output tokens on gpt-4o-mini = $0.000045
✅ Pricing table and calculate_cost() defined


## Step 2 — Build a Cost Tracker

The cost tracker wraps every LLM call, extracts token usage from the response, calculates the cost, and keeps a running log. This is similar to the structured logging from `mini-observability`, but focused specifically on dollars.

In [4]:
class CostTracker:
    """Tracks the dollar cost of every LLM API call."""

    def __init__(self):
        self.records = []  # list of per-request cost records

    def record(self, model: str, prompt_tokens: int, completion_tokens: int,
               latency_ms: float, label: str = ""):
        """Record a single LLM call's cost."""
        cost = calculate_cost(model, prompt_tokens, completion_tokens)
        self.records.append({
            "label": label,
            "model": model,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
            "cost_usd": cost,
            "latency_ms": round(latency_ms, 1),
        })

    @property
    def total_cost(self) -> float:
        return sum(r["cost_usd"] for r in self.records)

    @property
    def total_tokens(self) -> int:
        return sum(r["total_tokens"] for r in self.records)

    @property
    def request_count(self) -> int:
        return len(self.records)


tracker = CostTracker()
print("✅ CostTracker initialized")

✅ CostTracker initialized


## Step 3 — Tracked LLM Call Function

We'll create a wrapper that calls the LLM and automatically records the cost.

In [5]:
def chat_tracked(message: str, label: str = "", temperature: float = 0.0) -> str:
    """
    Call the LLM and record cost/token usage in the tracker.
    `label` is an optional tag for identifying the request type.
    """
    start = time.perf_counter()

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        temperature=temperature,
    )

    latency_ms = (time.perf_counter() - start) * 1000

    tracker.record(
        model=MODEL,
        prompt_tokens=resp.usage.prompt_tokens,
        completion_tokens=resp.usage.completion_tokens,
        latency_ms=latency_ms,
        label=label,
    )

    return resp.choices[0].message.content


print("✅ chat_tracked() defined")

✅ chat_tracked() defined


## Step 4 — Track Costs Across Different Request Types

Let's simulate a realistic workload with different kinds of requests — some cheap (short), some expensive (longer output). We'll label each one so we can see which request types cost the most.

In [6]:
# Short, cheap requests
chat_tracked("What is an API? One sentence.", label="classification")
chat_tracked("Is Python compiled or interpreted? One word.", label="classification")

# Medium requests
chat_tracked("Explain RESTful API design principles in 3 bullet points.", label="explanation")

# Longer, more expensive request
chat_tracked(
    "Write a detailed comparison of REST vs GraphQL vs gRPC. "
    "Cover use cases, pros, cons, and when to choose each. Use markdown formatting.",
    label="detailed_analysis",
)

print(f"✅ Made {tracker.request_count} requests")

✅ Made 4 requests


## Step 5 — View Per-Request Cost Breakdown

Now let's see exactly what each request cost.

In [7]:
# Per-request breakdown
rows = "| # | Label | Prompt Tokens | Completion Tokens | Cost (USD) | Latency |\n"
rows += "|---|-------|--------------|-------------------|------------|---------|\n"

for i, r in enumerate(tracker.records, 1):
    rows += (
        f"| {i} | {r['label']} | {r['prompt_tokens']} | "
        f"{r['completion_tokens']} | ${r['cost_usd']:.6f} | "
        f"{r['latency_ms']:.0f} ms |\n"
    )

md(f"### Per-Request Cost Breakdown\n\n{rows}")

### Per-Request Cost Breakdown

| # | Label | Prompt Tokens | Completion Tokens | Cost (USD) | Latency |
|---|-------|--------------|-------------------|------------|---------|
| 1 | classification | 15 | 27 | $0.000018 | 1627 ms |
| 2 | classification | 16 | 4 | $0.000005 | 476 ms |
| 3 | explanation | 19 | 172 | $0.000106 | 4622 ms |
| 4 | detailed_analysis | 38 | 1113 | $0.000673 | 30010 ms |


In [8]:
# Find the most expensive request
most_expensive = max(tracker.records, key=lambda r: r["cost_usd"])
cheapest = min(tracker.records, key=lambda r: r["cost_usd"])

ratio = most_expensive["cost_usd"] / cheapest["cost_usd"] if cheapest["cost_usd"] > 0 else float("inf")

md(
    f"**Most expensive request:** `{most_expensive['label']}` — ${most_expensive['cost_usd']:.6f} "
    f"({most_expensive['total_tokens']} tokens)\n\n"
    f"**Cheapest request:** `{cheapest['label']}` — ${cheapest['cost_usd']:.6f} "
    f"({cheapest['total_tokens']} tokens)\n\n"
    f"**Cost ratio:** The most expensive request cost **{ratio:.0f}x** more than the cheapest."
)

**Most expensive request:** `detailed_analysis` — $0.000673 (1151 tokens)

**Cheapest request:** `classification` — $0.000005 (20 tokens)

**Cost ratio:** The most expensive request cost **140x** more than the cheapest.

## Step 6 — Cost Summary and Projection

Per-request costs are tiny. The real question is: **what will this cost at scale?** Let's project costs based on our observed per-request average.

In [9]:
avg_cost = tracker.total_cost / tracker.request_count
avg_tokens = tracker.total_tokens / tracker.request_count

# Project at different scales
projections = [
    ("100 requests/day", 100),
    ("1,000 requests/day", 1_000),
    ("10,000 requests/day", 10_000),
    ("100,000 requests/day", 100_000),
]

rows = "| Scale | Daily Cost | Monthly Cost (30d) |\n"
rows += "|-------|-----------|-------------------|\n"

for label, daily_reqs in projections:
    daily = avg_cost * daily_reqs
    monthly = daily * 30
    rows += f"| {label} | ${daily:.2f} | ${monthly:.2f} |\n"

md(
    f"### 💰 Cost Summary\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total requests | {tracker.request_count} |\n"
    f"| Total tokens | {tracker.total_tokens:,} |\n"
    f"| Total cost | ${tracker.total_cost:.6f} |\n"
    f"| Avg cost/request | ${avg_cost:.6f} |\n"
    f"| Avg tokens/request | {avg_tokens:.0f} |\n"
    f"\n### 📈 Cost Projections (based on avg ${avg_cost:.6f}/request)\n\n{rows}"
)

### 💰 Cost Summary

| Metric | Value |
|--------|-------|
| Total requests | 4 |
| Total tokens | 1,404 |
| Total cost | $0.000803 |
| Avg cost/request | $0.000201 |
| Avg tokens/request | 351 |

### 📈 Cost Projections (based on avg $0.000201/request)

| Scale | Daily Cost | Monthly Cost (30d) |
|-------|-----------|-------------------|
| 100 requests/day | $0.02 | $0.60 |
| 1,000 requests/day | $0.20 | $6.02 |
| 10,000 requests/day | $2.01 | $60.21 |
| 100,000 requests/day | $20.07 | $602.10 |


The per-request cost looks tiny, but at scale it adds up. This is why cost tracking matters — a small change in prompt length or a shift in request mix can significantly change your monthly bill.

## Step 7 — Cost by Request Type

In a real application, you often have multiple features (classification, summarization, chat) hitting the same LLM. Grouping costs by label tells you **where the money is going**.

In [10]:
# Group by label
cost_by_label = {}
for r in tracker.records:
    label = r["label"]
    if label not in cost_by_label:
        cost_by_label[label] = {"count": 0, "cost": 0.0, "tokens": 0}
    cost_by_label[label]["count"] += 1
    cost_by_label[label]["cost"] += r["cost_usd"]
    cost_by_label[label]["tokens"] += r["total_tokens"]

rows = "| Request Type | Count | Total Tokens | Total Cost | Avg Cost/Request |\n"
rows += "|-------------|-------|-------------|------------|-----------------|\n"

for label, data in sorted(cost_by_label.items(), key=lambda x: x[1]["cost"], reverse=True):
    avg = data["cost"] / data["count"]
    rows += (
        f"| {label} | {data['count']} | {data['tokens']:,} | "
        f"${data['cost']:.6f} | ${avg:.6f} |\n"
    )

md(f"### Cost by Request Type\n\n{rows}")

### Cost by Request Type

| Request Type | Count | Total Tokens | Total Cost | Avg Cost/Request |
|-------------|-------|-------------|------------|-----------------|
| detailed_analysis | 1 | 1,151 | $0.000673 | $0.000673 |
| explanation | 1 | 191 | $0.000106 | $0.000106 |
| classification | 2 | 62 | $0.000023 | $0.000012 |


This breakdown is actionable. If `detailed_analysis` requests dominate your costs, you can:

- **Shorten the prompt** — ask for a briefer response
- **Cache the results** (recall from `mini-cache`) — avoid repeat calls
- **Use a cheaper model** for that request type
- **Set `max_tokens`** to cap output length

## Cost Optimization Strategies

| Strategy | How It Reduces Cost | Tradeoff |
|----------|--------------------|-----------|
| **Caching** | Eliminates repeat API calls entirely | Stale responses for changing data |
| **Shorter prompts** | Fewer input tokens per call | May reduce output quality |
| **`max_tokens` limit** | Caps output length | Response may be truncated |
| **Cheaper model** | Lower per-token rate | May reduce quality |
| **Batch similar requests** | Amortize prompt overhead | Adds complexity |

Cost tracking tells you **where** to apply these strategies for maximum impact.

## 🎯 Summary

### Key Takeaways

1. **LLM costs are variable per request** — different prompts produce wildly different token counts and costs; the most expensive request can easily cost 10–50x more than the cheapest
2. **Cost = (prompt_tokens × input_price) + (completion_tokens × output_price)** — tracking both token types separately reveals whether your costs are driven by long prompts or long outputs
3. **Labeling requests by type enables targeted optimization** — grouping costs by feature (classification, summarization, etc.) shows you exactly where the budget is going
4. **Per-request costs look tiny but scale fast** — a fraction of a cent per call becomes hundreds of dollars at production volume, making cost tracking essential for budget management